In [10]:
# import Qdrant libraries
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from qdrant_client.models import PointStruct

import torch
from PIL import Image

# import huggingface libraries and the models we'll use
from transformers import CLIPProcessor, CLIPModel


In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open("images/orange_cat.jpg")

# process the image so it can be passed to the model
inputs = processor(images=image, return_tensors="pt")

# forward pass through the model
with torch.no_grad():
    image_features = model.get_image_features(**inputs)  # (1, 512)

# normalize to unit length (optional but recommended for cosine similarity)
image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

embedding_vector = image_embedding.cpu().numpy().squeeze()

In [ ]:
def img_to_standard_embedding(processor, model, img_path):
    """
    Create normalised embeddings of an image

    Args:
        processor: Processors for an image - processes the image and converts it to the format needed for a model
        model: Pretrained Transformed model - A model from which to pass in the processed image
        img_path: str - Path pointing to an image

    Return:
        embedding vector corresponding to the image embeddings.
    """

    image = Image.open(img_path)

    # process the image so it can be passed to the model
    inputs = processor(images=image, return_tensors="pt")

    # forward pass through the model
    with torch.no_grad():
        image_features = model.get_image_features(**inputs)  # (1, 512)

    # normalize to unit length (optional but recommended for cosine similarity)
    image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)

    # store in CPU, transform it to numpy and remove any dimensions of 1
    return image_embedding.cpu().numpy().squeeze()


In [22]:
cat_embedding = img_to_standard_embedding(processor, model, "images/cat.jpeg")
orange_cat_embedding = img_to_standard_embedding(processor, model, "images/orange_cat.jpg")

In [24]:
print(orange_cat_embedding)

[ 1.08617526e-02 -9.06119030e-03  2.19716765e-02 -7.55415261e-02
 -2.98100207e-02 -2.19498742e-02  1.62419956e-02  4.72793877e-02
 -4.52281628e-03 -5.25705866e-04  1.92869045e-02 -1.72542557e-02
  3.26446816e-02 -2.92902924e-02  3.61027755e-02 -6.58178353e-04
  5.12280948e-02 -2.67243944e-03  3.16684581e-02  2.37116180e-02
 -1.09127946e-02  3.51148471e-02  3.85238007e-02 -3.32929268e-02
 -1.02611892e-02  1.80709399e-02  1.90085005e-02 -8.76461063e-03
 -1.61345918e-02 -3.67701538e-02 -4.70578205e-03  1.93744395e-02
 -3.49017140e-03  1.90506503e-03  3.27891931e-02  3.10719237e-02
  2.25570425e-03  1.95898823e-02  2.97576990e-02  1.46812454e-01
 -3.69019285e-02 -3.79781201e-02 -2.78692581e-02 -4.07910421e-02
 -6.92879083e-03  2.68935841e-02 -1.97530328e-03  2.46475283e-02
  2.82991379e-02 -3.65669243e-02  5.49744954e-03  4.58525904e-02
  1.60628464e-02 -1.48018813e-02  9.46559478e-03  2.27820058e-03
  4.41892706e-02  4.62851375e-02 -5.88031486e-03 -1.87634788e-02
  1.11239135e-01 -1.46939